In [1]:
import mlflow
from mlflow.tracking import MlflowClient

MLFLOW_TRACKING_URI = "sqlite:///mlflow.db"

client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

2025/08/27 21:54:23 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2025/08/27 21:54:23 INFO mlflow.store.db.utils: Updating database tables
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


In [3]:
client.search_experiments()

[<Experiment: artifact_location='/workspaces/mlops-zoomcamp/02-experiment-tracking/mlruns/1', creation_time=1755819227471, experiment_id='1', last_update_time=1755819227471, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}>,
 <Experiment: artifact_location='mlflow-artifacts:/0', creation_time=1755818886995, experiment_id='0', last_update_time=1755818886995, lifecycle_stage='active', name='Default', tags={}>]

In [4]:
client.create_experiment(name="my-cool-experiment")

'2'

In [12]:
from mlflow.entities import ViewType

runs = client.search_runs(
    experiment_ids='1',
    filter_string="",
    run_view_type=ViewType.ACTIVE_ONLY,
    max_results=100,
    order_by=["metrics.rmse ASC"]
)

In [13]:
seen_rmse = set()
unique_runs = []

for run in runs:
    rmse = run.data.metrics.get("rmse")
    if rmse not in seen_rmse:
        unique_runs.append(run)
        seen_rmse.add(rmse)
    if len(unique_runs) == 5:             # <-- te quedas con 5 únicos
        break

for run in unique_runs:
    print(f"run id: {run.info.run_id}, rmse: {run.data.metrics['rmse']:.4f}")

run id: 6ac54695e6c74b6b92ed495a187c07ff, rmse: 6.3086
run id: b26d1cd2a7e34bc4a64773a7ca6009fb, rmse: 6.3154
run id: 2c1e1834d3834d49827bf203607af429, rmse: 6.3156
run id: f74959bb0b044c95a03f02812564b40e, rmse: 6.3162
run id: 250cb6cf42384250bca183b023b309ea, rmse: 6.3166


In [14]:
import mlflow
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

In [17]:
run_id = "6ac54695e6c74b6b92ed495a187c07ff"
model_uri = f"runs:/{run_id}/models_mlflow"

mlflow.register_model(model_uri=model_uri, name="nyc-taxi-regressor")

Registered model 'nyc-taxi-regressor' already exists. Creating a new version of this model...
2025/08/27 22:12:12 WARNING mlflow.tracking._model_registry.fluent: Run with id 6ac54695e6c74b6b92ed495a187c07ff has no artifacts at artifact path 'models_mlflow', registering model based on models:/m-62ec9865895a4f19bd2cafecca0e2a1f instead
Created version '1' of model 'nyc-taxi-regressor'.


<ModelVersion: aliases=[], creation_timestamp=1756332732789, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1756332732789, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='6ac54695e6c74b6b92ed495a187c07ff', run_link=None, source='models:/m-62ec9865895a4f19bd2cafecca0e2a1f', status='READY', status_message=None, tags={}, user_id=None, version=1>

In [22]:
model_name = "nyc-taxi-regressor"

latest_versions = client.get_latest_versions(name=model_name)

for version in latest_versions:
    print(f"version: {version.version}, stage: {version.current_stage}")

version: 1, stage: None


/tmp/ipykernel_11878/47423457.py:3: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  latest_versions = client.get_latest_versions(name=model_name)


In [24]:
model_version=1
new_stage="Staging"

client.transition_model_version_stage(
    name=model_name,
    version=model_version,
    stage=new_stage,
    archive_existing_versions=False
)

/tmp/ipykernel_11878/3159747828.py:4: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=[], creation_timestamp=1756332732789, current_stage='Staging', deployment_job_state=None, description=None, last_updated_timestamp=1756333195460, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='6ac54695e6c74b6b92ed495a187c07ff', run_link=None, source='models:/m-62ec9865895a4f19bd2cafecca0e2a1f', status='READY', status_message=None, tags={}, user_id=None, version=1>

In [25]:
from datetime import date

date = date.today().strftime("%Y-%m-%d")

client.update_model_version(
    name=model_name,
    version=model_version,
    description=f"The model version {model_version} was transitioned to {new_stage} on {date}"
)

<ModelVersion: aliases=[], creation_timestamp=1756332732789, current_stage='Staging', deployment_job_state=None, description='The model version 1 was transitioned to Staging on 2025-08-27', last_updated_timestamp=1756333288272, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='6ac54695e6c74b6b92ed495a187c07ff', run_link=None, source='models:/m-62ec9865895a4f19bd2cafecca0e2a1f', status='READY', status_message=None, tags={}, user_id=None, version=1>

In [37]:
from sklearn.metrics import mean_squared_error
import pandas as pd
import numpy as np

def read_dataframe(filename):
    df = pd.read_parquet(filename)
    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)
    
    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)
    df = df[(df.duration >= 1) & (df.duration <= 60)]
    
    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    return df


def preprocess(df, dv):
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    categorical = ['PU_DO']
    numerical = ['trip_distance']
    train_dicts = df[categorical + numerical].to_dict(orient='records')
    return dv.transform(train_dicts)


def test_model(name, stage, X_test, y_test):
    model = mlflow.pyfunc.load_model(f"models:/{name}/{stage}")
    y_pred = model.predict(X_test)
    return {"rmse": np.sqrt(mean_squared_error(y_test, y_pred))}

In [38]:
df = read_dataframe('data/green_tripdata_2021-03.parquet')

In [39]:
client.download_artifacts(run_id=run_id, path='preprocessor', dst_path='.')

'/workspaces/mlops-zoomcamp/02-experiment-tracking/preprocessor'

In [40]:
import pickle

with open("preprocessor/preprocessor.b", "rb") as f_in:
    dv = pickle.load(f_in)

In [41]:
X_test = preprocess(df, dv)

In [42]:
target = "duration"
y_test = df[target].values

In [44]:
model_version=1
new_stage="Production"

client.transition_model_version_stage(
    name=model_name,
    version=model_version,
    stage=new_stage,
    archive_existing_versions=False
)

/tmp/ipykernel_11878/2444359950.py:4: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=[], creation_timestamp=1756332732789, current_stage='Production', deployment_job_state=None, description='The model version 1 was transitioned to Staging on 2025-08-27', last_updated_timestamp=1756336610255, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='6ac54695e6c74b6b92ed495a187c07ff', run_link=None, source='models:/m-62ec9865895a4f19bd2cafecca0e2a1f', status='READY', status_message=None, tags={}, user_id=None, version=1>

In [49]:
%time test_model(name = model_name, stage = "Production", X_test=X_test, y_test=y_test)

CPU times: user 10.4 s, sys: 40.2 ms, total: 10.5 s
Wall time: 6.17 s


{'rmse': np.float64(6.261709137119395)}

In [50]:
run_id2 = "9f36ca45ecfa47b7bfbbec8a70c2b86b"
model_uri = f"runs:/{run_id2}/models_mlflow"

mlflow.register_model(model_uri=model_uri, name="nyc-taxi-regressor")

Registered model 'nyc-taxi-regressor' already exists. Creating a new version of this model...
2025/08/27 23:20:50 WARNING mlflow.tracking._model_registry.fluent: Run with id 9f36ca45ecfa47b7bfbbec8a70c2b86b has no artifacts at artifact path 'models_mlflow', registering model based on models:/m-c8e8078a7a9a416e86497aca4aefcbe1 instead
Created version '2' of model 'nyc-taxi-regressor'.


<ModelVersion: aliases=[], creation_timestamp=1756336850175, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1756336850175, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='9f36ca45ecfa47b7bfbbec8a70c2b86b', run_link=None, source='models:/m-c8e8078a7a9a416e86497aca4aefcbe1', status='READY', status_message=None, tags={}, user_id=None, version=2>

In [51]:
model_version=2
new_stage="Staging"

client.transition_model_version_stage(
    name=model_name,
    version=model_version,
    stage=new_stage,
    archive_existing_versions=False
)

/tmp/ipykernel_11878/2006082376.py:4: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=[], creation_timestamp=1756336850175, current_stage='Staging', deployment_job_state=None, description=None, last_updated_timestamp=1756336886323, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='9f36ca45ecfa47b7bfbbec8a70c2b86b', run_link=None, source='models:/m-c8e8078a7a9a416e86497aca4aefcbe1', status='READY', status_message=None, tags={}, user_id=None, version=2>

In [52]:
%time test_model(name = model_name, stage = "Staging", X_test=X_test, y_test=y_test)

CPU times: user 10.4 s, sys: 27 ms, total: 10.5 s
Wall time: 6.09 s


{'rmse': np.float64(6.261709137119395)}

In [57]:
from datetime import date

date = date.today().strftime("%Y-%m-%d")
new_stage="Staging"
model_version=2
client.update_model_version(
    name=model_name,
    version=2,
    description=f"The model version {model_version} was transitioned to {new_stage} on {date}"
)

<ModelVersion: aliases=[], creation_timestamp=1756336850175, current_stage='Staging', deployment_job_state=None, description='The model version 2 was transitioned to Staging on 2025-08-27', last_updated_timestamp=1756337181404, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='9f36ca45ecfa47b7bfbbec8a70c2b86b', run_link=None, source='models:/m-c8e8078a7a9a416e86497aca4aefcbe1', status='READY', status_message=None, tags={}, user_id=None, version=2>

In [56]:
from datetime import date

date = date.today().strftime("%Y-%m-%d")

new_stage="Production"
model_version=1
client.update_model_version(
    name=model_name,
    version=1,
    description=f"The model version {model_version} was transitioned to {new_stage} on {date}"
)

<ModelVersion: aliases=[], creation_timestamp=1756332732789, current_stage='Production', deployment_job_state=None, description='The model version 1 was transitioned to Production on 2025-08-27', last_updated_timestamp=1756337147080, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='6ac54695e6c74b6b92ed495a187c07ff', run_link=None, source='models:/m-62ec9865895a4f19bd2cafecca0e2a1f', status='READY', status_message=None, tags={}, user_id=None, version=1>